# 🚀 Road Inspector — Train Crack Segmentation Model (YOLOv8-Seg)

This notebook trains a state-of-the-art **YOLOv8 Instance Segmentation** model on the **`crack-seg`** dataset (3,717 training images, 200 validation images).

Works seamlessly on:
- **Google Colab** (Free NVIDIA T4 GPU runtime with 1-click cloud download)
- **Local Workstation** (Windows / Linux with CUDA or CPU)

### 📋 Notebook Pipeline:
1. **Environment & Hardware Detection** (Colab vs Local, CUDA GPU check)
2. **Ultralytics YOLOv8 Installation**
3. **`crack-seg` Dataset Preparation**
4. **YOLOv8-Seg Model Training**
5. **Validation Metrics Evaluation** (Mask mAP50, mAP50-95)
6. **Visual Inspection of Predictions & Training Curves**
7. **Model Weights Deployment** (`best.pt`)

## ⚡ Step 1: Detect Environment & Hardware
In Google Colab: Ensure GPU is enabled (**Runtime → Change runtime type → T4 GPU**).

In [ ]:
import os
import sys
import torch

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
print(f"✓ Environment: {'Google Colab' if IS_COLAB else 'Local Workstation'}")

cuda_available = torch.cuda.is_available()
device = 0 if cuda_available else 'cpu'
print(f"✓ Compute Device: {'CUDA GPU (' + torch.cuda.get_device_name(0) + ')' if cuda_available else 'CPU'}")

## 📦 Step 2: Install / Verify Ultralytics YOLOv8

In [ ]:
try:
    import ultralytics
    print(f"✓ Ultralytics YOLOv8 version {ultralytics.__version__} is already installed.")
except ImportError:
    print("Installing Ultralytics...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    import ultralytics

from ultralytics import YOLO
ultralytics.checks()

## 📂 Step 3: Setup `crack-seg` Dataset

In [ ]:
import urllib.request
import zipfile

if IS_COLAB:
    dataset_dir = '/content/datasets/crack-seg'
    dataset_zip = '/content/datasets/crack-seg.zip'
    os.makedirs('/content/datasets', exist_ok=True)
    
    if not os.path.exists(dataset_dir):
        print("Downloading crack-seg dataset (~91.6 MB) inside Colab...")
        url = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/crack-seg.zip'
        urllib.request.urlretrieve(url, dataset_zip)
        print("Extracting dataset...")
        with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
            zip_ref.extractall('/content/datasets/crack-seg')
        print("✓ Dataset ready at:", dataset_dir)
    yaml_path = '/content/crack-seg.yaml'
    project_dir = '/content/runs/crack_seg'
else:
    # Local path discovery
    search_roots = [
        os.path.abspath('RepairAreaSegmentation/src/dataset/Train model/crack-seg'),
        os.path.abspath('src/dataset/Train model/crack-seg'),
        os.path.abspath('crack-seg')
    ]
    dataset_dir = next((p for p in search_roots if os.path.exists(p)), search_roots[0])
    yaml_path = os.path.join(dataset_dir, 'crack-seg.yaml')
    project_dir = os.path.abspath('runs/crack_seg')

# Create YAML configuration
yaml_content = f"""
path: {dataset_dir.replace(chr(92), '/')}
train: images/train
val: images/val
test: images/test

names:
  0: crack
"""
with open(yaml_path, 'w') as f:
    f.write(yaml_content.strip())

print(f"✓ Dataset YAML configured at: {yaml_path}")

## 🏋️ Step 4: Train YOLOv8-Seg Model

In [ ]:
model = YOLO('yolov8n-seg.pt')

# Execute training
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16 if cuda_available else 4,
    device=device,
    workers=4 if cuda_available else 0,
    optimizer='AdamW',
    lr0=0.01,
    patience=15,
    project=project_dir,
    name='train_exp',
    save=True,
    exist_ok=True
)

best_weights = os.path.join(project_dir, 'train_exp', 'weights', 'best.pt')
print(f"✓ Training complete! Weights saved to: {best_weights}")

## 📊 Step 5: Evaluate Validation Metrics

In [ ]:
if os.path.exists(best_weights):
    best_model = YOLO(best_weights)
    metrics = best_model.val(data=yaml_path, split='val')
    
    print("\n==========================================")
    print("       VALIDATION PERFORMANCE RESULTS     ")
    print("==========================================")
    print(f"Mask mAP@50:     {metrics.seg.map50:.4f}")
    print(f"Mask mAP@50-95:  {metrics.seg.map:.4f}")
    print(f"Box mAP@50:      {metrics.box.map50:.4f}")
    print(f"Box mAP@50-95:   {metrics.box.map:.4f}")
    print("==========================================")

## 🖼️ Step 6: Visualize Training Curves & Predictions

In [ ]:
import glob
from IPython.display import Image, display

results_png = os.path.join(project_dir, 'train_exp', 'results.png')
if os.path.exists(results_png):
    print("--- Training Loss & Metric Curves ---")
    display(Image(filename=results_png, width=800))

val_preds = glob.glob(os.path.join(project_dir, 'train_exp', 'val_batch*_pred.jpg'))
if val_preds:
    print("--- Sample Validation Predictions ---")
    display(Image(filename=val_preds[0], width=800))

## 📥 Step 7: Export / Download `best.pt` Weights

In [ ]:
if os.path.exists(best_weights):
    print(f"✓ Trained weights available at: {best_weights}")
    
    if IS_COLAB:
        try:
            import importlib
            colab_files = importlib.import_module('google.colab.files')
            print("Initiating browser download of best.pt...")
            colab_files.download(best_weights)
        except Exception as err:
            print(f"Colab download prompt: {err}")
    else:
        # In local workstation: copy directly to backend models directory
        target_dest = os.path.abspath('RepairAreaSegmentation/backend/models/best.pt')
        if os.path.exists(os.path.dirname(target_dest)):
            import shutil
            shutil.copy2(best_weights, target_dest)
            print(f"✓ Automatically deployed to: {target_dest}")
        else:
            print(f"Local weights saved at: {best_weights}")
            print("To use in Road Inspector, copy to: RepairAreaSegmentation/backend/models/best.pt")